# Root segmentation: inference on one plate

Crops the petri dish, pads it to a multiple of the patch size, predicts the root mask patch by patch, then maps the mask back to the original image size.

- **Input:** `../models/unet_model_256px.h5` and one raw plate image, `../data/test_image.png`.
- **Output:** `../outputs/test_image_prediction.png`.

In [ ]:
# 1. INFERENCE LIBRARY IMPORTS
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras.backend as K
from tensorflow.keras.models import load_model

# Patchify: Essential for splitting large images into model-digestible tiles
# and stitching them back together.
from patchify import patchify, unpatchify

print(f"TensorFlow Version: {tf.__version__}")
print("Ready for Inference.")

In [ ]:
# 2. CUSTOM METRIC DEFINITION
# Keras needs this definition to reconstruct the model graph during loading.
def f1(y_true, y_pred):
    """
    Calculates the F1 Score. 
    Required to load the model if 'f1' was tracked as a metric.
    """
    def recall_m(y_true, y_pred):
        TP = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
        Positives = K.sum(K.round(K.clip(y_true, 0, 1)))
        recall = TP / (Positives+K.epsilon())
        return recall
    
    def precision_m(y_true, y_pred):
        TP = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
        Pred_Positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
        precision = TP / (Pred_Positives+K.epsilon())
        return precision
    
    precision, recall = precision_m(y_true, y_pred), recall_m(y_true, y_pred)
    
    return 2*((precision*recall)/(precision+recall+K.epsilon()))

In [ ]:
# 3. LOAD TRAINED MODEL
# Load the model you trained in the training notebook.
# Ensure the file path matches your saved model exactly.
from tensorflow.keras.models import load_model

patch_size = 256
model_path = f"../models/unet_model_{patch_size}px.h5"

# Load the model with the custom metric mapping
model = load_model(model_path, custom_objects={"f1": f1}, compile=False)

print(f"Model loaded successfully: {model_path}")

In [ ]:
# 4. IMAGE PRE-PROCESSING UTILITIES
# Functions to detect the Petri dish and crop the Region of Interest (ROI).

def detect_petri_bbox(gray: np.ndarray) -> tuple[int, int, int]:
    """
    Detect a *square* bounding box enclosing the Petri dish.
    Uses Adaptive Thresholding -> Morphological Open/Close -> Contour Detection.
    Returns (x, y, side).
    """
    blur = cv2.GaussianBlur(gray, (7, 7), 0)

    # Adaptive Threshold handles uneven lighting better than global threshold
    thr = cv2.adaptiveThreshold(
        blur, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 51, -10
    )
    if np.mean(thr) < 127:
        thr = 255 - thr

    # Clean up noise
    k = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(thr, cv2.MORPH_OPEN, k, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k, iterations=1)

    # Find Contours
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    h, w = gray.shape

    # Fallback if no contours found
    if not cnts:
        side = min(h, w)
        return ((w - side) // 2, (h - side) // 2, side)

    cx, cy = w / 2.0, h / 2.0

    # Score contours by Area (bigger is better) and Centrality (closer to center is better)
    def score(c):
        x, y, ww, hh = cv2.boundingRect(c)
        area = ww * hh
        mx, my = x + ww / 2.0, y + hh / 2.0
        dist = ((mx - cx)**2 + (my - cy)**2)**0.5
        return area - 0.1 * dist

    best = max(cnts, key=score)
    x, y, ww, hh = cv2.boundingRect(best)

    side = max(ww, hh)
    side = min(side, h, w)
    mx, my = x + ww // 2, y + hh // 2

    xs = int(mx - side / 2)
    ys = int(my - side / 2)

    xs = max(0, min(xs, w - side))
    ys = max(0, min(ys, h - side))

    return xs, ys, side


def refine_to_inner_plate(gray: np.ndarray, bbox: tuple[int, int, int], pad: int = 40):
    """
    Refine bbox to the bright inner plate, removing the plastic rim.
    
    UPDATED: Uses 'get_outer_bounds' logic to ensure the top of the plate 
    is not cut off, even if condensation causes gaps in the threshold mask.
    """
    x, y, s = bbox
    crop = gray[y:y+s, x:x+s]

    rmean = crop.mean(axis=1)
    cmean = crop.mean(axis=0)

    # Otsu's method to distinguish plate area from rim/background
    _, rbin = cv2.threshold(rmean.astype(np.uint8), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _, cbin = cv2.threshold(cmean.astype(np.uint8), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    rbin = (rbin > 0).flatten()
    cbin = (cbin > 0).flatten()

    # LOGIC UPDATE: Use outer bounds instead of longest run
    def get_outer_bounds(b):
        indices = np.where(b)[0]
        if len(indices) == 0:
            return 0, len(b) - 1
        return indices[0], indices[-1]

    r0, r1 = get_outer_bounds(rbin)
    c0, c1 = get_outer_bounds(cbin)

    # Pad the crop, with extra safety for the top edge (r0)
    r0 = max(0, r0 - (pad + 20)) 
    r1 = min(s - 1, r1 + pad)
    c0 = max(0, c0 - pad)
    c1 = min(s - 1, c1 + pad)

    # Calculate new square dimensions
    hh = r1 - r0 + 1
    ww = c1 - c0 + 1
    side = min(hh, ww)

    cy = (r0 + r1)//2
    cx = (c0 + c1)//2

    r0_final = max(0, cy - side//2)
    c0_final = max(0, cx - side//2)

    return x + c0_final, y + r0_final, side


def crop_square(img: np.ndarray, bbox: tuple[int, int, int]):
    """Return the crop defined by bbox."""
    x, y, s = bbox
    return img[y:y+s, x:x+s]

In [ ]:
# 5. LOAD TEST IMAGE
# Load one raw grayscale plate image.

test_image_path = "../data/test_image.png"

# Flag '0' loads directly as Grayscale (H, W)
image = cv2.imread(test_image_path, 0)

if image is None:
    print(f"Error: Could not read image at {test_image_path}")
else:
    h, w = image.shape
    print(f"Image loaded. Dimensions: {h}x{w}")

In [ ]:
# 6. EXECUTE ROI CROP
# 1) Detect outer dish bbox (Rough location)
bbox = detect_petri_bbox(image)

# 2) Refine to inner bright plate (Remove plastic rim)
# Note: Using the Top-Safe refinement logic to preserve root origins.
bbox = refine_to_inner_plate(image, bbox, pad=20)

# 3) Crop to square petri dish
petri_dish = crop_square(image, bbox)

print("Petri dish bbox (x, y, side):", bbox)
print("Petri dish shape:", petri_dish.shape)

# 4) Visualize the cropped image
plt.figure(figsize=(8, 8))
plt.imshow(petri_dish, cmap='gray')
plt.axis('off')
plt.title(f"Petri dish crop ({petri_dish.shape[1]} × {petri_dish.shape[0]})")
plt.show()

In [ ]:
# 7. SYMMETRIC PADDING
def pad_to_patch_size(img, patch_size, value=0):
    """
    Adds symmetric padding to the image so that its dimensions are 
    perfectly divisible by patch_size (256).
    """
    h, w = img.shape

    # Calculate how much padding is needed to reach the next multiple of patch_size
    pad_h = (patch_size - (h % patch_size)) % patch_size
    pad_w = (patch_size - (w % patch_size)) % patch_size

    # Split padding evenly (Symmetric)
    top = pad_h // 2
    bottom = pad_h - top
    left = pad_w // 2
    right = pad_w - left

    padded = cv2.copyMakeBorder(
        img,
        top, bottom,
        left, right,
        cv2.BORDER_CONSTANT,
        value=value
    )
    return padded, (top, bottom), (left, right)

# Apply the padding
petri_dish_padded, pad_h_tuple, pad_w_tuple = pad_to_patch_size(
    petri_dish, patch_size
)

print(f"Original shape: {petri_dish.shape}")
print(f"Padded shape:   {petri_dish_padded.shape}")
print(f"Padding added:  H={pad_h_tuple}, W={pad_w_tuple}")

plt.figure(figsize=(6,6))
plt.imshow(petri_dish_padded, cmap='gray')
plt.title(f"Padded Image ({petri_dish_padded.shape[0]}x{petri_dish_padded.shape[1]})")
plt.axis('off')
plt.show()

In [ ]:
# 8. PREDICTION & RECONSTRUCTION
print("Model Input Shape:", model.input_shape)

# --- HELPER FUNCTIONS ---
def patchify_image(img, patch_size=256):
    """Slices image into non-overlapping patches."""
    h, w = img.shape
    patches = []
    for i in range(0, h, patch_size):
        for j in range(0, w, patch_size):
            patches.append(img[i:i+patch_size, j:j+patch_size])
    return np.stack(patches, axis=0)

def unpatchify_image(patches, full_h, full_w, patch_size=256):
    """Stitches patches back into the full image."""
    recon = np.zeros((full_h, full_w), dtype=np.float32)
    idx = 0
    for i in range(0, full_h, patch_size):
        for j in range(0, full_w, patch_size):
            recon[i:i+patch_size, j:j+patch_size] = patches[idx]
            idx += 1
    return recon

# 1) PATCHIFY
# We use the symmetrically padded image from the previous cell
patches = patchify_image(petri_dish_padded, patch_size=patch_size)
# Normalize to [0, 1] and add Channel Dimension -> (N, 256, 256, 1)
patches_input = patches[..., np.newaxis].astype(np.float32) / 255.0

# 2) MATCH MODEL CHANNELS (Safety Check)
in_ch = model.input_shape[-1]
if in_ch == 3:
    # If model expects RGB, repeat the grayscale channel 3 times
    patches_input = np.repeat(patches_input, 3, axis=-1)
elif in_ch == 1:
    pass
else:
    raise ValueError(f"Unexpected model input channels: {in_ch}")

print(f"Processing {patches_input.shape[0]} patches...")

# 3) PREDICT
# The model runs on all patches at once
preds = model.predict(patches_input, verbose=1)

# Ensure shape is (N, 256, 256) by removing channel dim
if preds.ndim == 4 and preds.shape[-1] == 1:
    preds = preds[..., 0]

# 4) UNPATCH (Reconstruct Full Padded Mask)
predicted_mask = unpatchify_image(preds, *petri_dish_padded.shape, patch_size=patch_size)

# 5) UNPAD (Restore Original ROI Dimensions)
(top, bottom) = pad_h_tuple
(left, right) = pad_w_tuple

# Slice out the center, discarding the padding
pred_unpadded = predicted_mask[
    top: predicted_mask.shape[0] - bottom,
    left: predicted_mask.shape[1] - right
]

# Verify dimensions match the original crop exactly
assert pred_unpadded.shape == petri_dish.shape, \
    f"Mismatch! Unpadded: {pred_unpadded.shape}, Original: {petri_dish.shape}"

# 6) THRESHOLD (Probability -> Binary)
predicted_mask_bin = (pred_unpadded > 0.5).astype(np.uint8)

# 7) DISPLAY RESULTS
f, ax = plt.subplots(1, 2, dpi=150, figsize=(12, 6))

ax[0].imshow(petri_dish, cmap='gray')
ax[0].axis('off')
ax[0].set_title('Original Petri Dish Crop')

ax[1].imshow(predicted_mask_bin, cmap='gray')
ax[1].axis('off')
ax[1].set_title('Predicted Root System')

plt.tight_layout()
plt.show()

In [ ]:
predicted_mask.shape, image.shape

As you can see `predicted_mask` and `image` have different shapes. This is because of cropping and padding.

The final mask must have the same size as the input and line up with it exactly, so the next cell reverses the padding and the crop.

In [ ]:
# 9. FINAL RECONSTRUCTION (GLOBAL MAPPING)
# Apply the corrections and map back to the full original image.
pred = predicted_mask
if pred.ndim == 3:  # (H,W,1) -> (H,W)
    pred = pred[..., 0]

# 1) UNPAD: Remove the symmetric padding
(top, bottom) = pad_h_tuple   
(left, right) = pad_w_tuple   

pred_unpadded = pred[
    top:  pred.shape[0] - bottom,
    left: pred.shape[1] - right
]

# Verify dimensions match the specific petri dish crop
assert pred_unpadded.shape == petri_dish.shape, \
    f"Unpad failed: {pred_unpadded.shape} vs {petri_dish.shape}"

# 2) PASTE BACK: Insert the result into the original-size canvas
# We use the bbox coordinates (x, y) found during the detection phase
x, y, side = bbox 
full_prob = np.zeros(image.shape, dtype=np.float32)

# Insert the prediction into the correct global coordinates
full_prob[y:y+side, x:x+side] = pred_unpadded.astype(np.float32)

# 3) BINARIZE: Convert probabilities to binary mask (0 or 1)
predicted_mask_corrected = (full_prob > 0.5).astype(np.uint8)

print(f"Final Mask Reconstructed. Shape: {predicted_mask_corrected.shape}")

In [ ]:

overlay_crop = cv2.cvtColor(petri_dish, cv2.COLOR_GRAY2BGR)
overlay_crop[pred_unpadded > 0.5] = (0, 0, 255)

plt.figure(dpi=250, figsize=(6,6))
plt.imshow(cv2.cvtColor(overlay_crop, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Overlay inside crop")
plt.show()

In [ ]:
# Save a binary mask at the original image size: roots are 255, background is 0.
mask_image = (predicted_mask_corrected * 255).astype(np.uint8)
assert(predicted_mask_corrected.shape == image.shape)
os.makedirs("../outputs", exist_ok=True)
cv2.imwrite("../outputs/test_image_prediction.png", mask_image)